# CMDO Stage U6 — independent pair-complete observer reserve

This is the one-shot outcome-blind reserve for the observer frozen in U5F.

The notebook first trains source models and computes target scores without using
target labels. It then commits a pre-outcome seal. Only after the seal is
written may the frozen witness protocol access target labels.

Do not edit cells. Do not rerun after successful completion.

Choose **Runtime → Run all**.


In [46]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [47]:
import subprocess, sys

install = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "medmnist==3.0.2",
        "folktables==0.0.12",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(install.stdout)
if install.returncode != 0:
    raise RuntimeError(install.stdout)

preflight = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import medmnist, folktables, torch, torchvision, numpy, pandas, "
            "scipy, sklearn, PIL, matplotlib; "
            "print('U6 scientific stack preflight passed'); "
            "print('medmnist', medmnist.__version__); "
            "print('torch', torch.__version__); "
            "print('torchvision', torchvision.__version__)"
        ),
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(preflight.stdout)
if preflight.returncode != 0:
    raise RuntimeError(preflight.stdout)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 3.7 MB/s eta 0:00:00

U6 scientific stack preflight passed
medmnist 3.0.2
torch 2.11.0+cpu
torchvision 0.26.0+cpu



In [48]:
import base64, hashlib, os, pathlib, shutil, subprocess, sys, zlib

EXPECTED = {"auth": "46ea98b060fcac734e88ae4cda07be18f1075602ffcc24ad52dbe778f0c87d41", "pipeline": "38a7d16fe9238de1c2f59e9c9f72dc75795d2daf83d96514b244d099e7cc0098", "protocol": "fc65b08ad9c10cc3ce138c620ee8acf13fb03ff255baf2deb1d9710c81dc810b", "spec": "cd88e72099be68258baa64fd7ff78137670b33b2deae3abf82a6a5d83dd2720f", "theory": "4b1b686deddc1fdb5362496d1875d59051e7736c03e00d2597b1b227c5877d0f"}
PACKED = {
    "pipeline": "eNrtfV1z28ix6Dt/BQ5Sexf0QpQoWbKsDVOXkiibJ/oqUcpmw7BwQHJIYk0CDABakl2+dR7v863zC/NLbvd8DzAgKdsndSs3W4lFDGZ6enp6erp7ehq/+7fdVZbuDqN4l8QfneVzPkvig9rvnJ1XO84oGUfx9MRZ5ZOdYyypua5bO7s6v3F6eTglzsOR8/f//C+nG4/JksA/ce7chlG6c5YslnOSE+dmmJH0I0mdO0J/OB+bjb1are0kMdnJZknuLNMkW5JRHsHLlFeaJKmTz4iTiNaTNPlEYieKnYfDi5Pa7Vlw2+7edc6D9zedi4vz7vW7Rq3WeSKjVR4lsZOkY5Ke1JoNBxpHk2eHPIWjHMDPSZgRJ4zHCMdZhiliHCHeUf78c22/4YSjv62ilDhZskpHxJmHQzLPaIs8TKcEasfLVZ4B/nNocNBw8jQEtDiCvNUiGYtWI6DECgjBW2ejBAb5c+11w8lIOBfFQIOcpD577czCbEYyH0HH2TJJc4dkebQIc8Jgjkk2SqNlnqQA6bBBcXHCCUCgVFumZCdZ5dAzoZ34MKgRyTLRGRvTz7WjhpOuYtqEoy/pHeZOOJ87w9UYGrBOU7KcRyPE4efam4bzmEY4Kmg7EnOdEsB+7DuLMI4mgDEbfxgnMbSbO3/p3sIkXSccjR2c9WgSjZx8FQOX+Vh1HI2hByd7jPLRDAphJgEsYsmGx/uCOW7UGAc296HCAqYgQ06aRUNAa9ygfFqDUS2cIJis8lVKgsCJFpSYYRwneYhAslqNlyHJ59FQPEaJ+PVblsTiN8zATPxOMvFrOQ9zYNiFeIZJGyfyKZsBS87l07NsBhNKxO/HMEUSyHefouUkmhM2AKBIOJqHWQazz9/LIlmDIDjtNX32aSefYKmxekvAHwYpqt3icOiL/HmJxObl7fjZd85g/sPhHGCcR6Pcdy6jDP69XwH5axo9lvMkB5CN5TP+ckKYhXku3serxfIZy+KlJBYQBwqw3pj1fdu9FB13FzCjrDQbRcvnBvADFon303CVZVEYB0Ac4Aa9JmUmYDJekzwto9x4DxMuyQeVwxS4VED4AGIhjRvzKIa/AV28omrv3fkZEhoYlRSqL0ieRiMJNE1GQbgaBXQJF2oixCAD6TOi4klwAEqOABZUHmSwtnKz0TJaEsRIzhZ/LtSCZQk9w+rWJhBWBlA5Hfdg2QHWtdrt3c2/d87unZbjnoGoyXauknE4D86jcBqD5IlGAZPT4TACPJ7dWu++/a6D1ekiezgKNAkfoIQPhIQPhIQPuIQPUMLD4uv8+Ra6BBENkja46F63LxHe/v7Rm9FeeHj8tjkaHoTk7cGbN2+O9g5BQJFxOGoevD0ejybh6GByMHp9uN9sHu01X8O/o+Zw8mZy+NZVcC/ubv7SuQ568Bj03rcR+mh8fEze7O+9fTskR8f7h8fDMDx6PRm/mUzeHDcP3hy92RseHAz3xyQkB+FwcrwfHoWH4+OD8Xgfmk006Denvc7dnzp3QfccIVt2HBhjr9PBt/t7+0d7b/Zf104fzt917ntQ1D/2neaR7xzs+87Ra/i9fzyo3XVuL7tn7ftOjzbaq121/xz80um+e49Ts9c4OKzddXt/DM6wi+5Zt3ON5cewYZ53Lu/bwf3NPaXiXqMpik4vb87+SIv29g9r77u9+5t3d+2r4LR7jZ3ES+TqbBmOiAdVAA/853WzXmuf9YLezcPdWScALKDq0R78R4vv23cwCl58SItrvOzupnffucPx1Rz477M7CRfR/Nk9cdyrzjmM7TI479xd3fTObm5/dX3HZbIe32N5Ozi77LSv6Qvc21Bs4rsRMHPsfvG/Euj1TbfXCfaCveMS5DiJMhLA0I+/Hvzp5cNd0Az2SsCH81UaILd/Neh37Sv4txm8LsGehotFCMBffz3wf7/tvAsODkugf1uSKZZ/NeDzm1+ue+2r28tO0CxjPk4e4yxE6YBvLZ1ct+8f7qCT7hUIGRP+WfeifRfAEoE+3r2UWbaGCyvk1x4Ms1Omeho+Zyg1vwm+4MfmXiU/Nve+qQfOkvtVLLn/TdAZ4+xVMI4Vc5Qbtw+nIN5AeoHYhB4CEIuvzW6qagX3f8aKuEkTrAeP37+Pi0ujD3j8/n10zT66/x193LaNPuDx+/dx897oAx6hj0GNbg8XHeQjuoexXcCFZrdYv3f2no7/qn1HHzt0Vs+7PfzT6dE6Z917WqX7jv2BBhzI9RmWAJN2/9S9p3LnvMMAnXd+pfx6fte5wr+33euzWwaTdXQvgFx0aMFd+6wJFRDjm9vbm173HpWZz267jaM5PcU68C9SgBKzTX+f0t+ntA6UfAHF6X8qTZv+69xT2p2u4vGcnNA+Gc1PnCxP6TOjrnpmFt8JbsWolIGAocXUCAvmSYiWqtS2+/2Br9UcMAjUqETl8sSZQIuc9SOMw+KLbLWkxVOYPHuDNMo+BKA2Pj3r75VVeUJ1/j6MwGfvB9rIgiz6BHCjONeRi8ZqxNSEziJUdVlhrTYmEwcs0iBOHr26s/MHLGbkSwkYZ7E0WRpYQ1gtDWhSb0RZggIozL06h5TNwv3DI7QCiIc2zQk1ZUy4M5hwbtQ1WH1oji/AtJxRS6iRgELruenQraM9MgvVnNJ5BdNzNFvFH9DrAEZl6s3DxXAcnvCajZSEY68Jq8V55eCfuu8MXbeuIFA0Gqsljs2joOr6kGeNGXkaR1PQ/4sjo5PvfQznK5NzzCFCEVPxQuCxOI+mq2SVsaa0UIfiO2Mw8kgLXtAZPXpdN5ExSQVNG3kyfAbjxKvXLYhKsz5AA9lLhr+doN1onVus0RiDLZhhPd8hcYb2OGAdRa2LcJ6BkZkhX34gz1nrPl3hM1mGaYgejpbn+rguT9x6gUo5eco9/IdymbXrwrCwcoPEI7DHPJc6tVzr6GBNRmDJZGCTeK8AkTxTowPGZ10gMDQNTk7cxm9JFHvQuYeV65R38BeyDm1u0BogeC9FrH/SPBqgSVF3fnC8/VevDvYFsvMEvTK4nn8D+zJIkyRnawwXBUNVelYyKbTxP6zgubvIPGDX7Y5x0LtXz+f4FxbFrsPtRn99i94sTMmY/s7KzZjsoKtJ+neALAoltWCiiSpukKcoy4H7zPXESSir0ZcgG0YzNjZc2Jz88AP6sWHs1hvpdJ4MPY5oHXumIiHKgnGUenWGNJTOQURw8HWn1XKaCh2OCn/b32NtwKbPiHMBouk6yS8S2Cg6aZqk3sQ9o34nZxVHf1uR+TOfN4fPm4PzduJ85uC+uGJ60afwgaQxmXvLJAvK2wlsGKCWlcspD6hHhjiAEDKDiQcFU4gIKh8YwwJgs7bqyVKbE8STFELg/RPfuQZZPnD+gOD6+Nt3TgZ1gIkAPA0A/vcTmLKHIFHNtkD5TY11egF3BB/DNApjsHu/gmrUzdWnwMUOeCI4LY2egCbWWdFhMqSgGlSmEDzWtrEA68UTwjd5hNf6ixC4vtVkb0fJ3PZ2T7bFMUrwMAR49KAYpmacTABKXXAwFNaB/k2HgLBFX4GAb4MAxWUIUGiBIGhM0XxiHgaB167q+SfZ1a4EZrAMkMmXwPg8LsMIpArI4Bj2Bk8wr20mBatWvkvxDAMKmWe28Y7EhO4tfq1itn2U0YP//hWDSsoCdg4kCoAFDQJ/QRNOHtY3rgQYQmNJ0sWKOa5lC9gYZoOB3jUsE1tlClRVZloB1EcobGkWFpWxEvkCtK07jQcMNpJ6h8ZJszILcQ5gTVkbsUQUS/jOzOSKWUImEzybCtJwHK0yj7GJ0k3pxFKghj6AxJaiBn1h8gF9+43sbyldp7PGPJl6+409YFjNz4Z7Gy195Wj9AaKGAHok0XSWB+imDVajoWdQiSvcrMUwCrMAtHUCgP5mvOE6vFGGWrssqBgfIyNvXQc8jQErd6MatyA8Q0/J4V1Njq+rx2Q2e2uMp1il5Nl8xVvhuIqVm2SnuS/LCgRGHgjEqVMgzqy+TUTYrKktp4I1X5FSyw1iR9lYoFr+A4QNl+QoNZx/a0lJg0dssvwHZx/f7WlaDtVn/oQLkykyLj3glYd+8sgwJfTgNHPgbzh3yEcS43AiPNjdhY5CesI7Ao0ozxqo3Wwp4LYWbqziLJxPnJYa0e6usy85I/TpnyHv8wQrD2hZH3+eyL6wJv4Z8k5FTfzNa7IVDErcBxzCZ0kw5mTweHcUVt3X3p4W3g71t6ey7dDS9vS08Fa0/aKQYVxYtN8RQ72SEkfVNZmIy/QqVTul1gp17zjEs0dvqdQioRShRs5o1gCTemGo9xL9PrYHYpuoslKuepkKXrkfCZQPQjY2tYpySx9XLWfOyWo+D6hqQn9pG51Nx6QzwlcaDygIAGA0xtUATYaZJ/dHusHNwb7x5KAbbO+DbQ+2yh3ZN0eFyhcqXrO/yY3WM4QWNBJSCAT/K2efuzvobpStm2gmsYdoq6ytN0pgnYdTg2mGSTIvcdeUJHg4+rymIt8/gSrBNFxW9So5isKVzEN5vDDFAavScoTDr08LBsVqlMy+E/iiIIY2gkt0UANNBKKKoZinSvmIFdtRknLljh654fTrGNDZ0mYP1S0GjU2ehMTmD0BZ1Qpz7cilwgbgG1UoSmYR39rMQtzbVIkaEmUufIlKo9ECRwjjYQgy9GFzt6JU2OZZk1evsL7O4pbuOR9zOJIeJsU5E6tKtFTWEQys3iNXemyy/tAyUKgX5JLg6UJbRZXft+xD5trM66JMEryvAG7kEbYeVCyQvu1QKCfmXAhC1OV0oHQtzMRPRdK+Mns1HKLldajtPwIx3RJQkk6irUs6XXfVtlBRF7Y78VPbBsewCkewEriIhEpSUtsriRkRNaVZoaqXRDbULZVp9XFcASOce2IbLieqPli9efhkaw029BaNs2ixmudhTJJVFgieBjiUIcP53BNlWnutOZu6OAlmYboAEUbJXLk1Cb6vGgjSwdD77cD05bkOlLk27LDMOmVoX7idQGOHggkJMfwr8+jjZg+Z5lVnSjVrV3SgH+xLrRqd5RiphEbygdqXGCD4t99oNLgjS2+SzcIl6e80BwW/okQgJUvAHX3xvnMAWwg6f3a4b0gCB0Nt//AQFj2HijxEPTVgu1NDGwqZVgtGtO88+s6INWT9C4RmUvVHe+BxK0OAhms5M7ZBYazfYzTOZ85ileXOkFALABV9etpFHV3YK8wE9ushOlQ59/F/j+rnqK75ujwoECcVeDbPgagKgh7TJz5pIEPhzziaTLwpdfPRagAmXGL8UgsL0a8I/9tpngwE7Oe1zZul5tCWOiIlamPcFgGLnwBWfdMoq4aIpVlpjF4Tm7AaWT4WFeBn+T2XpTCGURKjfznGwyc5iX2YCB05IJ9PsS8VUlR82p+mSHBqMHNcOIP0JcF9sMvl/Dlg609GEDAsyqtQM8CxGj3M4aY3IWPqzeEW88uX6goaH8uFKvvAFcfjSEr+fCTtKFk+e3y7feLkLg9WLD1h7PNVy6x9IEMIUjqAcg/HwaHpWGBQYppjrOvM44Eh+vFhFk0XodxLtVYYK+i5ULffHCitgnPwaB4tPWRDtJNjPDedM78sBYd/PpHWE1v8MM0iKKxeJAOOBA+qnkEfoKPUp5uTdf2IaCjKNwyIVg9yNDlxbZmDoH+giLoVn6T4fc1kXrmySadCIKn3xGu1tD4rKSKI/KxR76uJxMK8dCrRkhdP+zJ5hHHI+ddQ8xnE7zPBdOGoICnj6O7JtgnaF5YcKxPpApUnQ7D7zgdClgCSnQmvo7FlaGzvZDJc3zzXTweNcNJnA31ZoAICkojgxtlIVvlyhdZaf2DozTyMmRLBJA4TkukUPU30t/l2oipI8u6bEIpQ1OhlqaaBlFUJaUGsJhNqtEZJ4xRP/rs3nlmDbvgNtEA1cYvw640s/Eg8BsF3WLhGy8U4MtcXNGzxv7ZeGyAiP3h75itGzAbbefWQBoYHDd9gzeu424HOCzNy9+7UFYaFyRIwz6MPHgPKSfDy9aoFN5qi7RP5/4FHlhH6sqr5wHQvLPAOSQsboXoBJPI8/JdtQiCYoJCSssUA3rHHCDau0+5l97rTvqsXBEmWJylBVYvCllDV+JlS3YQhFMv2Bl/RYyULClS+mdNKavXEfYg/xMBmukr0Wf5WgQGTKOcqFo/BotcMPKFNRXGwRteCt+waUPkt2FOVLfFdVbuitsacxeLagm+cvRtn6vS+RUtecNC0VdPN5LJNxy/cb/B0K5JWLLG+O5J3OFy/9Na441FuzOyXLGu582Qa4C8LDKqOzpezsHVIdl7b36PRj4Fkrf3DvT17lTyZt5pk59D+lo6C+w1a7jCcoxNjXIEN00MDGsPZwrkpVyvQTXsc1JT3jc5PA7jNKxjUOpPVfYOr6pJbmFsdXaEUzBLWTDTKMVppGBYBKs7Do2pQX+RZdaAHUhg3bTzFkr7WYV0q/aL17/E4u2jO3q1iDDYUK6/Hru8xScz4MkkjErOjJuBtAIaRju2Hs9ZnDviLa1hetJWOiS8wEMFrWBpks2iSByrk0tODKSsPKXkAZsXrUpxo6aBRX3b8qqJhPxn9W44Q+QVCo42BlKUNhzmj98cC1hYfEtDNFrxH3xlGcdYyr68ALBJnqDjQGEEdgypo7O320DTccE/RnnZpQEvT1wsb2WohInc0PKCl/iRbaoV6S/TNzcOl4SSN4mixEsTgg9Oa1/X2zE2G7CNBoNeCVeehE84Ox1uEUkjnQXVLdCdoDekjbweKGyyTOSk0lmtJeOcMacK9KSWhA+UCnpyv/l6jiSbLIf7zdlAvNdoxmgmmWdusXjhE4CSg6w5P4qnLpzoehDvS+WwVDmV+0mbh1Su9XJLYLDYpCO98HSctYFtnC2rKCXZ5xSJUyNPS21Gd6xZ8vTLKWwKFuk0MXtGIwAONmFgmYxBJ6A5i8k4/6NlrvMa4lwJZjLC9faygcDPeNfGdSQWNAuZBpo2/KC2UgAMUrCgrbnj7Vp/28hEDH4F7IsZS8kQjjvBaPegueDHN7omacu29OVKoZBbokLS5QGDao15LYxGspT1qtWxzD7VtxdZW7DDFeDY96vxyPFA8RTcJXb2Zl4aPGDWrReHjZeG+flWC7zn06uqCjBcxSkp+W/UcgV1dg7SuadopdXOqVx41ploufen6NFq2hQHXvHNYCGia4X0K5i2QmkMFIHi1NRh6RbcKIXi3FaCSXl7YRvmoG9FiqqlPtqpsVFpFip8VKENcr6opabi+bRhwFU66hMEIpL41acJgfF3BHCjA4ihuB0lpmb5T0EC0Fd9aa/AUiWuqo75GS9/RNUY90h9v8/rijiO7G+vK2GLmq0BezpTpTs32nCzQajduyBqOMazRF3ezBnisYrtnaRrs9EJHvCL64JgBSM3fNQ522UAxhanmM3TUxb5CnECZIrwBuzUGlrV7f9e+7l3c3F25dVukwEtUfm1Yhs5vXkhC5cyuNhf5pcQ32jk+mz1hzBuD1mVVWWVhU9cy57FsTjEKtQrkKlVjOLbYHzOgtlxZvybWYheQnHHW4gv7xLGbq9pKXLMKy/agviwtiEuittRPCxn0/aOl5qpf2GlspNH2NaOlsf8N1vWptriKrrWt0QJItdGaV001Df9tYaghNzoraRaNW0yu0L0juOpctq9vQMr8qRfcdaCAS5tye+0uXWviXpExBeAcNEDd0nYjutp/dj5T3vtRLqofB18KMItaMVeM+NoobPOjaBKmX7vNA91Gs4+AO17+CvMwIyoBB71u3Nwr7fe83LPsprQWvxy2doveCIPfOKvenhHbis0ZX+kbKadNRW3+VsewDJpv0RpkLLED5nUlXA2TRUgjs6A2kDz2DPTAUjrwncOBBr9YXevTrC3VCkWbvupxoO6yUhXA6FavhwLnsKxA8HWkOhFE6ksseaRtNBYn/MDcU0LDfoWvSFetxk98s6f1iylWlLjEijVjr6SLGa0YVQ7sA2bF5LnF1YWa1blm0SHYfXqeX6N3e9m9d+t+0af2bdoOG3xfDlsTZgxb6yveilOo3Kb8YsPo1Li+s4pkZiv47tqRznj/0o/+qfQjLn5aplBDWdKSUqVKcxKiSwmtf2lJ/1AtiW/fNBcJaEeYjmRL5UhTOIQS8H3VIhZNA4MJwhHeypmRReiNJyfOctw4hz3rIg0XhCpGegGTXI9JivvteKKHQIDgw9wVDl4LBvGIdeD1fLWIWco9eNm+77jFd0oWYmkfQdCAaP6ATZiQWEQsLRgIY9aUXcNmPwGokc3jJ6fv3j6cnt38iabY+OXd/S2AxSvZrL4FSRnOyDsqnq78kTyLkxXoCtNkvRbXhMaOGKho63zmvwoHKtih1EuzIByHSwzZsdKdHTkal3xVLg1jIirmku/OYSHK3YOe8iSIVwsQ/iOPUZpmPQFxT3CIWcsdJQSY2K07v3eODhVD/Q97a5bDxN78AHNubYbApsgC4Q8tp6l7P1mUE90a6fTNkxETcDozii2Hqh6sfl9nkEGDbqsmKuXOZcwcUj4E3YvW9QR03hZ61Q+KYJ7CuLVDb0rgnbF4wh9iMhUPDPIzvbJqUENiK9jXQhLVLbvFX9ZEObeB9vhcsIKQQb7SBpok8w855nWRlg9QFFmWHTTWzIM4451ivmyVfiTPwTMJ05bLcvjId7MkjT6B9HObO7/Ce+0Na9VylyTNklh7gXYRjsG0jfTjCCaMx5OqZcJPiFC0o6ruUS04a/Xd61+R9qZhZcB8kpoHjeTVlrPsVAXg0B2Dt8OIZjNjnSZr1kReWpRXDUzv4ZSlMXO1MA5paGBKEg+jKEezJBoxi0PiwwMpTZx8miQ1HBF+xqipfLbxi8I+VbjlG/pY+xqbxrSOJK71CkPn0GLoCCS2N3V0elYYOv88wR3NzcEdKLz/scEdFZNydXPeuXTr3x72IflUMygVt6rCejmGocLIkRCFuambNqYVbIn4kF1LY9XsVMV+aKC2Cf9AJYVL4+qwDwXzi/vdjd7KtG2b7F+maVfL7FpZ3S4LcIYQy/o2GFj9bIU7gazbJxmtUBTrEq+6PnCUTqKpkOwq6WgheHiNdLfYJZttc9aPJvorDIGNe4EcgbYXqFFU7wWVhBOFfD8Qb9h+sJXzQKL0L0/BVp4C5vRqCUpzdfiEl/PHf5n439nEr8w6ef3rtra+NCeZ3uvcPlz1HKbq/ixT7aMcE7Y/ffjG45BwPv8KQ6C0OSzpXQe3/3A0cG5TzHOHNnr5RNj4mID+BQLO71xyicVInnJcjGtDNERaISsGhsP12zq3Hhyt7byKJ74ND4vpxrEgT/jNCWoS972COCpsGvXq7ZxNKE+BwmAxTBoMmM8R4yc2DBIropeoGdbSjSIBgSogENygstzr349AXwrNH9f4a9zh7VufBaQvf415rnYoFF0VnC0m48Pmwr8jEeB3JJg4Z2Htivl9fQJOLMuAJ8Nh394IxKXqk0L+GyNovPAKzRO+mpQsCtLksaBylSirEY9eQP1sLH6VLtecNLOSzJZrTGWhkpRuqp4sKlTVBJmqrBVau6cCtogDLbQjwiK5TExKe1Qp8ktALweAVQWiiR6s8WjrYtJKXdlC0wwCcK0F02XSODktUarAorj3fNGnXyRi5ZUVI9UtahJlLqHfYA67WpH7qKqtOyC9Qut6sQXNSNnS1o+zq76vAJLwhi81tniCc/WRl6Adj4N7SSr8wEJjlH10yzihnwveeIVuMUvPmDyJ0Gd+XQOsJ1jhZvKiLGe5BOgHIPQARKyN+hVNB/1wf3Zz1QlAXF+fB7d3HVHQ67QvtR3WHaUkxAhNGJl7ohIAazWKYgGqFYu02uwzNTLlF1tvtk826OGNjJxMQkJ1Q4BX1SsxGs01W0h6a0AyUyyw7+ZAc/5FCGPE4js68Fp9EUKvwY17sGJwJtCO0fJaqJmdsHhSE1GaFLkw/fXyMNlyYh8agsafC3fyNCmzdq3Zxa5agOWOmWodsC8S0QGyeIwCCzb3Qfjk6N4sVfqi2BcwQ4OmeooEkws3JKG5mNBiUq/km75b2PAEcQc0RRB7prW9jYu4h9XpQkU83HqDfi+J4SiHqiVGZt9piugHVjBbgZkPWZtAmiQYNKYWTxPsl6OcGSyJMNvL01UcPEZ5DFQP+Ge2vO12bgGIX8zXtmld/PlO9dNA5CjknF/YulkOHVm0zXYuI0606BjOmbqd59XFhRTp5zds6HLrymssmuuCx9OJJHp8KRTcFjZl7RIb7rBvfEHLKSbL4BobHfBnY+F9cY2e8fyNOngB24Dm2KNXCWk2YYGSr4+oj/cCBvVtEONapMjcMYziELN3VeODOeikc+RxRlKBAjtdEcmQRTa9yqp7RlV6kihT/2ErlRGVPuHhGN6u4VKVZffbOL6yF8LtxtlqMolG6ORzhK/V4esDRrdczZnvzzItsAFaIIpUh63POvpf6BFWqF6IkXxxC3aomWer2vkpApcNV6dp+dPDL55PyqbS6ZniCum9NK3HwO9z2eu+jfK8tQJtUXW1lWapK8gd0ISSvLpkmnJ9MQtGfTEZlvov0aO/UpfmzVZE3rtYbQF8mLH5NS5q0BLbKF6orX8XjZ3Nj30nPZG7idnmS82yEtgeMGYfXRSalLnaMakU+zBHS1SVCT91MHLrQUjsiEypXvXyXfkXeZ3tnmeT2zl2vsKkfMmtXDKaJRnouyxLquZ+5ozOHc+SCEWfcxU8lkxVg8cXwovhgQQCqvDMiFUpee1e274a28Dfoh6gWFFvu5Wm1VzP/1aQ63hdF9oVp3Xx1N9itrX8eYywfZUIb1B20NIEd1rdYlq8gWW6dAXMKuirBf6LBf+LN4CC+UQbjNfXlCOidiP/vQ6VbcTtN8r0b5Xa22VBLNUuZ0VkJWuaAFrJfAXsIHBDjUFyoaYlbO7WCoqz6HaAFiGwJZmCDRN8zNgwWHu3KlZTHg1UYA2/7VhUQluLn5HxUSw6vXSwdnRPlsaqcF3bqnyPAoz9/TqI1hSQApzl5TpYtsyZAlT53WAThYu5JA1Cmy83wiolkzSAFd6unYGv0KO+qz5l6kgFXUl5lIouSVPgiyQAvKZXK+0LWWOaJqvlEIxHIebV9+J8KZgHvgPzYLgUBahGOC1oSLCuW15R1PhshorRKVyYsCZ2yVLRslJ6AKBqyVIFTC1qbK4t8crenzCBg9ZGrWyfLv5iC+uahYb2tVzRbXmltjzb8t2AdWmVApTyyrWPo7wsBcXMtbqO0uZCFO0Ly7MCgJDp0Eru7VB1EqVZXq6rn8d7he19i1ZiFzfbyr29CoJxmO+ZwmRzr9o5vmeXJTYYluQHapX7XAiYXjpPugDkd9ZWi0WYYiQRE5w8m4cCZIZg+5p8sb2h8PlXs4vvt/QqmgeD3K8o/AYU2+evkGz/L8qzl4qgfwaBojOtOal9d4pRhgXxPTDmuthCER8q7pRe0/LKDlU4Fi9oLAg8eqa/jDOzGcrar/SVVTvH1nmGNjqbtnQYbfIPbesKMuspK1NToJK4ZV1mWtz6Y8udk0nuGrcHWZNtV7Lq4V+r9xtXr3FbxJiEvmtXncy1V2wjFl2xXF+T+sSjvsT9dy3Dw6++kyX1Vvajz/9InRSPExQY42pLOksC6arKlgQoEcaad0qDv9E2MOraltnAOJGbJcqPL7HABJF5BBYrN0WX7HMPlorsjfxYUjInYySeqipXRb/I1APjS4O8raK/FYSdyU1IanF+Lh3uCuc6zYNakLdCCDRifmZlnJhzRlkHQMRn2QEo4kBz9VCuobHgSZks5fq48cjvELDsV9pE7JZB6Fg9Jqm8g8o5k5siEqQpBwQTVposA5bhv2ZxWfDeOCnXdbPlEl/TlTx/4eF1fM687bbtPzh7Iqfdhm9q6Nxp+FrkRwKLabsqnCVriF3hP+E9WIfPM/YVZvYbu4s29bbmyx3WnmyenHXdVKlvm3i1qtk6BqJylB0UhMMI80IHQjKjo2aWrK/LBCOuYPZBJy0yhJs7Jif6BZ73hRzjps4i/ECCSTSll+7XxDmusW6M/r7K/sHky8ykSdIxv7BZ0EJpXAj/4ol1cYm9AjYPNhwP/tBjHe8t3qg8bIhPRZTzafBe6+JD6WMasLbXODiWMIdh6j3hB4boS/o1C97K3Nx9VqNF//VZvEbLvQBEHYmoCfMnK8x1wGgwmPwAoQbwCXbWDxlmwZdwlKWXJuxaT+vNoe/MwpabcsVsksQ5JdQbBemZ9uW5VyAKHLEtstulWn/A9XPiuV1QQfFEBWMKHo74zO3MyUcyx4NKmhU9lndkseGcTPEERoeE3/iah8/AfVoxZleHaSyEHV3Q+Q0ejoKmCBq8ancay3gKoxkvo1bzeE/BGM2TjHhr2OMY2eN147huzoyFhLLIKt1V8/BpRu/80RSV+CvLn+ek5e7slCfrq6blnLn2YdTO3//3/1HfooSC8vT0YKHtcCXTmJtVTmWK+83TsC+m4R3QZat5EOLfUKaZjFFmDtdwX2Tm4Fb4tXaE2G62ah9V2hE2HjtqHCOPHWk8tpwnufxiVV9zMasyYzT4CkThB5K23MSV0uCKDuRFUAvDrALMqrk690ULT6Rd3dc4mV5h9fAmKTQfhhlp6W85z54Wo404bmXu7mlEl3iX+fpmSZUx4G36cbZREk9wKx4RW5vvJXEOgjMOPDh9Dk7pELbieMPSW2/KSSPQMOZsbPWG7Wx7GlvBXOS5HoKgOt5o4+lVN5p4GF4S0M+eY2yJaoofOE3Rm6p/45RKxThGWVe4HQc1q666efSdHWfezoZkYVUrOVpcrjp30o2Ofedyh3UG+3U2wns68ejZCXMRZtPcP7awLAarKt/1i3ZMVPIcQ8n7doH8OjhF1fFGh/q998cqO9cvexx1++X7bIB0ulhUvEZ23kd5ByRz2C7xM8yrjDjJBOON2Q5KPjKR8e0kP9QuKvzCP2O4ieJUAx+vUsyUAepjHE1AU/CKCnhFZh3yNJqvxlRX1hwSIiT7nEO94lDV5Qm/XPlMBo/fkREoORmr/SlauoaBUbp4RK92wOpH7ZyMNcwb6XSeDD33lavH4UYT2qIRZSxcv04vttEi/E6xSLUjRnZSFBTVoTv2sB28ZcGiH7EPmrI69Wh38kWeaFjXK+IgXGTDYPjM7i5QACi6PbDlbZeRtBzXljsKhYsJ9lNu4wDLPNrWzqkWYUST3usWFP1oD7OgogVp4D/CG8hC+mSoHv+Wuoz1K7x4DNM4iqdZg+WZEY+eG03jJEUlCN0T0yR9bl2sMMfNL6yGkCSwrpNRMhfXf5CTvSRrkPhjlCZx3z27Or/BhXN7d3N/c3ZzGdy279+DfMB0rcn8o8Aab0NshNF+uH9f0T6fEUBxI4T7952bu18rYPDrN2CnjzYCuri7+UvnOujdds4qoC15ShADVMButQR6ZTl+qESvTGBwf+ZJikDVcMyuVhRvR9RLiJswiiPaDKp4O8kUO3K27Syv84LuDOFXXT7RDcDeVo7WuMzDJtXaQJtww3EnqG7HUJ+TevnmFSWUtWWRknXTJcOIlnwQ301WX5SEcWEmCs9dHQVF8tFkTUWK90tUVmdQKEdNiIXhVkEsVFsD0aR5BTyzUhW04nU2JF40QXenxgbKzLP2ZJmYqu6WYQqtgtXhBKYshl0upbucMRR5me7h8CK46F63Lw1g2/VvANIFQe992wCnNWYo6jf7THy0y33rQSxJuohyvHU4mqFnKxiSCd51W/ERutc31x23gkT6JTV+R027hxYk8fw5CCd4QbAQ4K7f1ckcVIMrerDcbqtjE2rSa2YFv/GjVs7ai+Duw5Goit/vZUTZkf6QKIYdKkVpNQlhsY4bwk2IycBgSYLhiEFisK5+Qw8elnIhPUoTIMEiGdOLc7Q6qHt7oFjBTizUJBfLzmjNK6zJqIt8huphywAir8shf1EuvBFzfpESAvoFXnZll+lSMo1YZipYDqiKKcAi1twU5yphn+hc7+5MxKfr9/LWCHzdl8GWjkV+6cjwVf2Va6sMKWP6OqZvcz5/qeuFePxEgIPGpnzYuFzKncTkMRiC7BurO5kFhqxui+Jw7YrZHtQL14WckA3L4vCCV5Vfcy8uAc2QKXIqvQQtkzMopZ48AVsadr0Yy7jygreF+XR7QAKwAK8a3JnslK5+5DeHtf4ZHtNVjPSDNT2LhlHORixPscPRh9XSQLaBH7gM0P7wLJ9ApDWocfKT44JCd3ffbV8GLjzhpXqqYtOr3WAJpBOqbLs//PrD4ofx/Q/vf7j6oefa0kfpdhKhPTO86oWJaSw+wL8em8iMJ4GnIwUGMC61ozHGM4aAAaUicQzNyzeUj0KRpjapwqJyo94YChMrHmifBZ2hb5l6r/blV6sM/uCpuHDw8oOQj5wZqULs7mKyLxj37mgxTmAXC8hyRhYkRRkTPor0HazRekJpie71JCydp3AkNxq6OnHVTMzTFQf+jyn0JBfpENo03QqmceG05xc8tXQtUQxjzmRuHr51ZtAtGm/OKsavuSroKmnOmuQ7WiIDX78PXkxboggubmXRm02mJmNNTrPD4bCL4/CLaRYnruqv3K7H6zLh5hSOqcBYfQYJ9ugwWSkIwjNOuaWQbhWrqR8f4l5sudAtR2dit+VRKDpk7RGf26BUs0aki4QUljVv3JzH+2NSRP7Ch3UnwUi3TWP6SfPcaKchviGLAXvMA91yp8p5o4e/b4lWezWO8qCHTQAZShCbA6mERr103rtlh/z8CFdqcMVavqRD86z4RV32WKMX9GZy0Za9XbBGL+uNHxxoJ/T6kpbMWOBynfwcxtR2SaIqI2nN5kajMoMrHyDedDNAKtmWSEqqM2pWMPP3CZ0Sn2gmAbFqUZU7em3xjk1ckdz+s3Ar/6jHXv04+PIznZeI6FX06CpLErSNGVYj5acXsiYIQemD4eRBUyiBWXAQiM5dW047w0POY70GNFv1EaVBIdRLr3Fgo0UlDUQa/58rSeBUILqZFuZVV5m3Zs141wTIOL93mmSnuW8bHR4lyQBqNZBKaD8OThoH5OWTW3G2W6pYcZvXON2oiG3CGdxrvN2zZ9bVZ35DYBWHdHy4KaltNcITiqlO0ErEkaJHky8/u1WAGLI6rLXoc3juCzLyVqU8JmlO1bFCTNiUJCjx1rNjdTSZmKi3b60MWTncMigx1JcOTIskASJGIA7DCbEurw3suCbyElYdjnJv73AzP9pCKte0fwEXGqhpNK1GfBM78uSqRVAG8t+PBTPYOwV+PAYWT8/iCRgKSbpBJGrRupSaxRfG3Y2fqmRkFWmFrq2RQnW4iYr82mCpqUJpEwSMiiq3x9LvSH1xlCs4RQQxrSV6IWaXLve3BQGsH1JjoO72ZFcpcLSxm13Czrtbrbm4awS2uE5RkNus9PvRtRxquo6i64JYuSx9c7g9AUVbbZBretjEhsv1YFjgLAUyfTnlLG5D4Win7nDfYc4Hlyrybqn5CJguQtdRkIERO5pFNERgfRuLi7C6yUA3COmXZloYr8iuvyzDjLUWomJsxvRMdRvRbsi8Q9zvaWyBMmIsaRZpsq8Wg6gMjP/FnhlCA/YNvk2XyDZR3KIl2IhskSUWulbc8WLkkQlrqVmDI+wLkg4a4Xzu6cEOYzJi32RqFfjexWyRAfWxAj271+ed2w78c30f3OKRcfeucx68v+lcXJx3r98ph/bZzfVF9+4KXppM6+Lh981dt9dhvvXgqn390Du7697eB71fr+/fd3rdXnBzffkr6xMGfHt387572r3vnLuFERKYwG8agkT3+uY+uHi4hF4rEb/r3Le710H78hI/Bvpwed+DVvDz7uF6A6rC2YJBPy1Y8q77O4cyKHqE//6f/+XoAVbinMW5YxZcrXbOx3Xi/MdnMcYv/1Gr/e53so64N5mdOJVyu7YjrSm9VsHiwlqmKwy7tR1YAAr08+6G9w0rC/uY43irYqqhgXSwsf0WY4B1bEwF4O2E4gPK5g6PgFrbytz7WWPeALd1Swu52/+AdakiJoKOeQAd08f0pus1P9Yrg8RIWwmipPEJjNlmbJvSF23UAAs331LEmQ6wsDsjBjhnZ8xwYUfcAk5mC3QtAas00djouC1SoLIV2kYjzYTI4mrjZAetGwetGxs0uw3EITGbnQZW7sijS3kiJSx4A+wGM59S0whshH5wm98xtnmnx7UFHfYGpYJzyw5VEDa0M7SIWu06ceSm48hNx0lGo1WaknGDS6fmPnf8G+dTNZBf69Kr3hGV/ngxrsipyqTh5nypXHTSQ9qtsyBvzmcsxCimX+I/9eCb9REYJ5Yj4jXZj63hIieVkRc6Httn53P57ONL7ll9eW7lDapLOQewTXMp16KHnJV5grdKJszgsGNVQGaUxGOMbdRiBp0dEUuoE/AZwOEkZ89ZA+aiOM/zMKffjj1xxM+G+OEZkVGMDTbmMVaMKsPQ6HEvz2Wsvdbe9q2hCAP6gb94czZja7jEhjTGrKtvSmTMnf0sXBcHWB0XXDfqbnkCsU1EcMWJx6douT6L+5oAYn6lDhrDE0bMNf4SLS8wck5ABWvkEaNJteMrWbV7G5x3Li7bsKzrTpg5YQpr4qOmon5FFPL6SGTMbiwwK+fH5P0zdvAY9lCGx9itjcHFipgG3090YtTXcqYkM9AQ+XQR5S/h0k2pbSvEvxLEVtktF5XdEuMY82GzJvzB1kCcNVfvNVSIFD5dLZZd4XNf+hL0bddx1y5Gfrj917gl/2OkQSvj7Aa/onXfcdRLM0JAGhl4BYD/Nirc80/d7yorwvWrTov8qkMiHaQSxNLm2RX5MvGu4K6hvBsfzbT4J6tfGxdeK2vRBAKGe6GI5S8W82C3oOTbsFzn466sXvBlr8NLXPFiWvquVIh3TZXYitu64yBb9Q3nPmuaVOcrtI/qtmAG4cBwfNyGsbND0XNaNWIznaMdgdNqLR1Z04rBWk/jdvX5lXkLau41eXSofuYI/cxh+hkzvWE5ajdOeZuzsp5vryhVfqWD2SvScFCh2YDCCjISK0q5WuyeSVQHNketNhepZl0hTJ08wTpFDYJVk04/2MBgMJ7uy8ObJrBXBjQ+LghoHHEQoAUTBPzbjOwSSu3/Age/MOE=",
    "protocol": "eNqlVtty4zYSfedX9ONcRA4v4s2OU0VJlM0tW1KR9CbOVsoFEqCEHZJQAEozytN+xH7hfsk2KNnjyabysg+SKKK70ef06QaKMrlN4TGA//zr35CtFukmxa9VCZsky835+mFzn5YprGdFmv89zSFPxwfY5OtyPV/fw9GxbMMoyqR8LIz1Y4kuqTm7x1CwXqVmcbcutXGxSedlho6XAJaxXt0/QXmHm/tLc5mvf0lXsJk/633TxfPdOl0uF9nq9tvWWQGbNH/IyjJdWMZqDWWS36alqUNny2wO5eMKHcalebJaZIsEMy9+ysr53cv7PM0fV5AsS4x3AZetV5bOH2lwXJ3qXTbLxi2MTZJrKjBBWGar5B7d5+t8YbhuENY28aPYqSuPsNgLwzCw/bpmjJLa8eKI1g2pvcarp77rOIHtTPG7dqombPzYMC54X7G9YEh0OlDcJabrB0ZNo4iFrh3HFQsi148qQoJpQ8OmCSPHC4PQrjyvcikjzCNVE7kkID6NPEpddGsM40wRIP+I2HDM4AoWTHbkYZUVJXSsJb3oiHlkUh2UKZkaYNBfrD9yKfqO9YO6MuqWkX4Ct+SgFCc99IIrBrZlR29eVu1BAophAlvSdQQfpxPjb5v0Fn47kJYPJ/D8CVDxpVek27fMHITpTE3S03FfIRkyHpqOcwVIRZKbjg01GV5yo2L7F6ltJTmpmrTsT7J07P/N0v1jajZu7rimgwQl8wJc252CYz4xImFzqFpew1xgJmTLrgwlDrJmsHq6hoHILRsUlD9PYHk/gQw/mwQQFKzvMCRKMCmwAChHKNbL8idUFGS6wbIySwvDxBrQrudquLnxLNty8Q1SUe+OXHHRgzz0A+/YmREkRDQNrzlpgZKBKDZAKwhlEr0a0X4eSNUydXODlbEcHUpDUQd5ZCc4aSga1gR2QvLfMfgZ3wT2yDD+PRtiL68f83kKD+tFeq8zzDpEDQ3peMuZuoJiQHhE0kLzLeEjFLeLeUuQ34Yz+a4V2+dWKPUei80GFBvX8HhtnfP5v/yLM/MtqVirgEimc+/4MDCql8uxGt8tS7HjFcd10Ey2MOzYK6l7yUxxGGqBz4qRFutV5smq2KzzEtKizB5wgBhLKX5nyE6NCjUppiJ5dRh0cShTteR7rBcKcYcrAmXYgdZJS/YTLC0KTu14M0xAXUCblB05Gd3HlVEqWK1PtuXjJzZQkTpRdl62DHXY74UcnrdkYHDzEhw+APu6f2fqLZ5Hy/eWMUjSq9FYcvX5GbF/PaELqsFB+5rs90jDeVfWM7k9vXUhhxptz9Ie/5jwD9ua2h/eOeZl0/cfUaD2h297fjR0d314Sfn88tcJ1C0f9xqEjuFPEFcc/4r04ixepUXxenoYswPV/XMFOEmcYAKeO4EAJeq4kWXkbI+Nh7hx3bWxQ2cE51WNgfdC8YEf2aeeIS/4AONM4f0WvvBhh0UFic6kZnpMWMYSYQHWGccOsrgnXOIgEPVnBUkygWQ2gdm5aWczy0hJvTsvA1fwhfHtbtSP0uG1fi7zC7XwT8H7AcR+zIeZZycdXhPNeiWkZWzOf+8EaxqqQ1DWDuR5EANpbzR/15c3KOXnMYLuX9e3jAfylXeHDl6LdEkGK+rhco5FRlBsHAkIEyJse2Ml4Egk1zyB1oxl/MKkuHgqVA2ph/ak6ZGCHtCoObQtYpGsHiB5nOtjL0fl509wi/LXA4BgSzGCo9cJXgeeJst7HQrXaDVGHsGbr0xzinnp+YoTnlOctfCDw0zHvR4Hn+4OBIjYeyYOCrGcJyz8iAzE9miEzT9SMG5rVqNcvjeM/LeG5xL0wtwR2YHUTTNGi2Nt9QVbdfhDMMm2En9+0Kzb/jerEdvpT5b3QrRY0IckxX1GY4ayQDCaSTgzef2WtvjTG+J4h8SjYDWBIwWXmv444r2k1rIja6HiRJmiUkweScXHc6rY48Tu0EuDClEERvpzOn8cbw2XY+XJ2PA9w15gL/eIK8OLSEidoGGx60WUObWLtxAW13ETurQO/TD2qUtJgzeHOPCdaeVOpxSvHSysa9uOIxyMOybk6VvEaeVUQRRQRmntNLTyvcCdxgF1otCnfmz7DgtDL6htj9k2df04RAfXDWs/CkM6Xk2+G9YdOUGFrOAVSilkV/TIPWlw/v/lyB47VOoDoDc0oxwZvuSIttic+mAw8MRBseDtjhwGffjhBlfGMrkv8Br6XxglfBQ=",
    "auth": "eNp1krtuGzEQRXt/haE2ccDH8qXOiNexGtlQZBdpiOHM0FpgvSvsUgmcIP8e+gkVccEpyDvnknP55+T0dIEwUEdQOM6/uoK7briPcCi7cepmpsXyNEM/8+dn6cRVR/FQsO4vlFD2TLgz1WxlsxR+KewXL7XS9pMQSyEWz015Gn/zEMc08/STp9g9MRc3X+PN+WrTXsSr6/by8mK1/vZ/+bxn7HKHULpxiPMOlLFPACTv2SkRQmLrlfEJwDaZXM6uXsJZJ5LWSREDa0jZK7BgyGsiVdvyi9seJh5KPJgcczdAHyfGcaIjH6WsQwHGB4lJAwftnLPCIDIToNTBE2ZAnTU2RklphWxqRZmyyya8+Lxiy+Oen6C3Nq7WF+1NW8t6Gzft93Zz18bz2+3V9Wb143y7ul7HOxnFW/d0GCLkUueB48O+5+dhfJDSXOCepfrouMB0zyX2kLiPUJ8xz0fSOA7946vVfuKxRj0+1K/B0FdKmQ4vkION+27PfTfw0ay0B0fSZg5Ke2KJqg6AA4bsFKEzLhhSBLmGEKyRTVJNQzVBdohCBL94Z09jGXHsj9gZrUnCAwWUAlEjS+3RKsHsAbPUOQmdszImQa6pJ0nBVaWXVJdI7+yy43F6PCI3SSbrLTERykzJaKuaYEl6Z8gEYSQ7py0KzUKQMsHVBqUcGu8c1Z908vcf/W7/Vw==",
    "theory": "eNqFVcFu4zYQvesr2F30ZtmxkQRbxzUQ2EnrQ2xjkz3saUGLI4sIRWrJkR3vIkA/ol/YL+mjJDvOpb2YlsiZN3zvzWjyy3w1e/q6vhMFl2aaTI4LSYWlJJYiK6QPxL9/qDlPP33Aa9ZsaDp7mK/E7rJ/Jf7562+xsIoqwo9lsdoE8jvyYuZsrn0pWTs7GbRhySTwIa4bpw4/c2c5zWWpzWF867U0vSBtSBGv8xujLaUF6W3B42H/6uqmlC/pXisuxr+NLqoXPPuttuNL/BeyZneTOeP8+OPoYjQcXb4mxbAFCPoHjUefqpfXYnT+ZnjKkbKrmhM3G+cV+XTjmF05xgkRnNFKfFRK3VRSKW23x91LZEyqntFnSYeotOLXjXHZ8/faMf3sMhrKOUYc811fX5/yNXvDJt1k0PEzGXQqRKKiJsP/oNwdKc/eUY6YZFJNQbl3djt9ZLkl8eVauJozV1K6AcVKeGqCI3JzbDKopgkAR9N17SsXsIP/MRFCdRBckABIYKHPSuiyiNz55sSppNy7H2SFzBkP7KFvJT0iEkU7Mq4qY7S24svVfV8sWChHQVjHuEsZT4qYsLYkMmmVVpIp9MVTRLDmICrypWYmlZwAUeJ69m19u/h8N//25+ru/n6+WP7Rf7vWOXMM/YlF40FNobtqHbvA6BNzD6R0Jo2AkKULmasO4xNdYo6X8mG5eHwSJRlpXSlT1BHqkIIUFjWQfBL0iyjIqBTkC5l9r3XQUSVBdqeRKNIQUCRQ30EvJdce0LqEeOEMdra4v/2cDi/ACx/xlNu2cBAIWhzxkp0ONXL8D9TaVbVprCNAszzDup09CnTVpVjX8EyGzgYg6ulFQdGmKgq4pH3y1flnAZ1aoXPyHnvs2nIqTzvt6gDZalsH7ARu5OxKGUTaG4VW7wzaNNDJhI+u9hmJ0ikyoXcUEKKA7F67ikKGgt42FYXM64qdD73Ogs5zAnHAKgpoK4aluvPehWjWaD6U0fpLbCiPqbsjRm4AD0YbVxeUSCvNIWh4cxV92dm9kByvnXYdJwJBhlIe2iZqO2Ov2VIIOOfYYYQlMsvicwtx5ts5ZTAN1IEsdQZbvHXm+ahFK8JdYAOiMG295kMPAljNlAZZVibe6qhfM6j2OhBaLsWoL5MtoVD2MaY2JlXIlMU+0UYEmVNMVjlnwIh1NtUWImvXggQyOBulbllKakZX8aHhd6Nl6KaC3DSv+2LpQPJGQxJ/EMOLX4Vs60p1CTZ21MwGLnCXwhmVoLOjbY6UvA3ZhoNbEeqGOdQdpxzklTBQdELkNlRdbTAhWh8WatQpQUwJUTpdO6GSQDZgNMehEwXsZlsk5CgXbu6OjO8LDVYjPZDe47OJpAhGsQhJTo57g5YZCsKYoz2Aw3PrQFhaghdNneiTwbsbDrovwaD5Sv8Loj/DGg==",
    "spec": "eNp9VcFy4jgQvc9XqDiRDCTYYAf2ZhOyk6oZoBIyl61dlbAbrI0tOZJMkp2af9+WJePNHuaSWO7269brfo8fnwgZsONRwZEZLsXgNzJITqDYEYgpgBxko8i+lNnzK9dAdKG4eLZB0IZXzIAmr9wUBF4aVpJX4MfCXA1GFnXPSiYyyKlmVV1ycURsoxpwQc401SC0VLbknVQEWFZ0sNiJqzoiDZbFN1kBOaml5oafYCzadk9AMlnVTHEthSYHJSvCjSZtQRjnXP8tuTBE1u2H4CB9e5kUB54DdkhVUwJ28QNfd601dQ2K6hfbXMXFMBgRttdD1zFlTTY2igldS2Xs6eKzYjlv9MVfYYuOMDmUhlGL0lZFoMnVJIw+RI00rGwjgX/vYGxV/aLMsJTHYXj9P6iL62F4KWjNuLLsti1dXAwQ4Gd7sxxOUMq6AmEwWkJm6aQVGMUz3d/zwMpyz7JnqnCMrj3fRAVMUM2rBssKkI2mmXQ70aYtFot5cL5Jm+wG70CCeH4zuZmF0SK6meLDrEvkAiErdwcqJC2YqrriQV/cp/2qvk+tpSyRgRx5yAytmL9FPAsX03AWxUGwmCzCj8lHxoWfxTSYRrN5FGJePA/j6GNiDxfOJ9HNYhLcRHEchsE5ze0iNUwdwVhig+6qjnULAqaQuR3ndkm3yf3D6pZ+2azu7m7v1793m+IA8I6NsBQGsX//KpU2Hp7um9z+Q50q8ERPJrMIW4qmcRQGcdTP3/Fhl9tw826r71DKKBNTYEM8I3ZmRB5IkoxIko5ImhAmcpKmTiEkeVpqJ2ntXKApS9LpeYyCF6C1zSLwxjJTvntNVeztv6swbSkdyL0GhfPDjn7FRJ8nWGW5Hyy/3W7IFtd8vJRW0ajgjZfyOG0b/SLhcMjRW8jGf+yhfJpftorV/d4nicVO047+JG3PSXdO23hyjqdtHL86E2yV55Dt1P84w/aAPVQPgg9/+s8VzsWb7SNaozkbWzuGs7l1RDs/04SLHGoQdq7lO56M9L6bXKekYOUJtB8EekXFjd3ArGDiiDzAQSqgTWxrrjfrlctD53zGxUMSecahXb+5U+Kgdw5tmHGetHtI1o/b5GG13tHt5nFHN0+75ebbij6uvq6Wu/vNmt5tHujd0+7pYUXv17er7Qr/YPb35Ov9bWIzXOETbqPdJtRjq380I+1+Gdz+dJ480OjE1mRtHrkk1ovbxSLf3QTINRl2j5/JB+/G8/yyt+n2rrWSb+8YCGAchGiarqTj2Wusn+ncDa8T5NRbSexlHoTz80z/ASX96tPOWG37K6cPpyAnTKubq8Gnn/8CGz9GtQ==",
}
decoded = {
    key: zlib.decompress(base64.b64decode(value))
    for key, value in PACKED.items()
}
for key, value in decoded.items():
    assert hashlib.sha256(value).hexdigest() == EXPECTED[key]

work = pathlib.Path("/content/cmdo_u6_locked_release_v1")
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True, exist_ok=False)
paths = {
    "pipeline": work / "StageU6_Independent_Pair_Complete_Observer_Reserve_v1.0.py",
    "protocol": work / "StageU6_Independent_Reserve_Protocol_v1.0.txt",
    "auth": work / "U6_INDEPENDENT_RESERVE_AUTHORIZATION_v1.0.json",
    "theory": work / "CMDO_v4.5_Independent_Observer_Confirmation.html",
    "spec": work / "StageU5F_Frozen_Observer_Specification_v1.0.json",
}
for key, path in paths.items():
    path.write_bytes(decoded[key])

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["CMDO_U6_PROTOCOL_PATH"] = str(paths["protocol"])
env["CMDO_U6_AUTH_PATH"] = str(paths["auth"])
env["CMDO_U6_THEORY_PATH"] = str(paths["theory"])
env["CMDO_U6_FROZEN_SPEC_PATH"] = str(paths["spec"])

print("U6 protocol verified:", EXPECTED["protocol"])
print("U6 authorisation verified:", EXPECTED["auth"])
print("U6 theory verified:", EXPECTED["theory"])
print("U5F frozen observer specification verified:", EXPECTED["spec"])
print("U6 pipeline verified:", EXPECTED["pipeline"])

log = pathlib.Path("/content/StageU6_v1.0_child_process.log")
with log.open("w", encoding="utf-8") as handle:
    process = subprocess.Popen(
        [sys.executable, str(paths["pipeline"])],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        handle.write(line)
        handle.flush()
        tail.append(line)
        tail = tail[-500:]
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"U6 exited with code {return_code}. Full log: {log}\n"
        f"LAST LOG LINES:\n{''.join(tail[-220:])}"
    )
print("Stage U6 completed successfully.")


Streaming output truncated to the last 5000 lines.
  5%|▍         | 7.96M/170M [02:50<55:41, 48.6kB/s]
  5%|▍         | 8.00M/170M [02:50<54:57, 49.3kB/s]
  5%|▍         | 8.03M/170M [02:51<54:32, 49.6kB/s]
  5%|▍         | 8.06M/170M [02:52<54:30, 49.7kB/s]
  5%|▍         | 8.09M/170M [02:52<55:03, 49.2kB/s]
  5%|▍         | 8.13M/170M [02:53<54:12, 49.9kB/s]
  5%|▍         | 8.16M/170M [02:54<54:05, 50.0kB/s]
  5%|▍         | 8.19M/170M [02:54<53:59, 50.1kB/s]
  5%|▍         | 8.22M/170M [02:55<58:01, 46.6kB/s]
  5%|▍         | 8.26M/170M [02:56<56:35, 47.8kB/s]
  5%|▍         | 8.29M/170M [02:57<55:57, 48.3kB/s]
  5%|▍         | 8.32M/170M [02:57<55:07, 49.0kB/s]
  5%|▍         | 8.36M/170M [02:58<54:48, 49.3kB/s]
  5%|▍         | 8.39M/170M [02:58<54:25, 49.6kB/s]
  5%|▍         | 8.42M/170M [02:59<54:11, 49.8kB/s]
  5%|▍         | 8.45M/170M [03:00<54:09, 49.9kB/s]
  5%|▍         | 8.49M/170M [03:00<54:05, 49.9kB/s]
  5%|▍         | 8.52M/170M [03:01<54:04, 49.9kB/s]
  5%|▌       